# Length-Scale Selection for Synthetic Data

This notebook performs hyperparameter tuning for Kernel Ridge Regression (KRR) on synthetically generated data.

**Workflow:**
1. **Heuristic Range Estimation:** Uses median pairwise distances to estimate length-scale ranges
   for target correlations (\(
ho\)).
2. **Data Generation:** Generates a synthetic dataset `(X, T, Y)` where `T` is the treatment.
3. **Hyperparameter Tuning:** Runs LOOCV grid search to select the best length-scale (\(\ell\))
   and regularization (\(eta\)).


In [5]:
import sys
import pathlib
import numpy as np
import pandas as pd
from typing import Dict, List, Tuple, Union, Optional

# --- Environment Setup ---
# Mount Google Drive if running in Colab, otherwise assume local execution
try:
    from google.colab import drive
    drive.mount('/content/drive')
    BASE_DIR = pathlib.Path("/content/drive/MyDrive/Colab Notebooks/CTE_Codes")
except ImportError:
    BASE_DIR = pathlib.Path(".").resolve()

# Add project root to path
sys.path.append(str(BASE_DIR))

# Project-specific imports
from KRR_methods.synthetic_dgps import generate_unified_data
from KRR_methods.algorithms.length_selection import krr_length_selection_loocv_joint

print(f"Working Directory: {BASE_DIR}")


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Working Directory: /content/drive/MyDrive/Colab Notebooks/CTE_Codes


In [6]:
# ============================================================
# Utility Functions for Median-based Length-Scale Estimation
# ============================================================

def pairwise_dist_median(Z: np.ndarray, max_pairs: int = 200_000, seed: int = 123) -> float:
    """Median of pairwise Euclidean distances (with subsampling if needed)."""
    n = Z.shape[0]
    rng = np.random.default_rng(seed)
    total_pairs = n * (n - 1) // 2

    if total_pairs <= max_pairs:
        G = Z @ Z.T
        sq = np.sum(Z * Z, axis=1, keepdims=True)
        D2 = np.maximum(sq + sq.T - 2.0 * G, 0.0)
        iu = np.triu_indices(n, k=1)
        dists = np.sqrt(D2[iu], dtype=Z.dtype)
        return float(np.median(dists))
    else:
        m = max_pairs
        i = rng.integers(0, n, size=m)
        j = rng.integers(0, n, size=m)
        same = (i == j)
        if np.any(same):
            j[same] = (j[same] + 1) % n
        dists = np.linalg.norm(Z[i] - Z[j], axis=1)
        return float(np.median(dists))


def solve_z_for_tau(rho: float, nu: float = 1.5, tol: float = 1e-12, max_iter: int = 200) -> float:
    """Solve for z such that Matérn correlation(z) = rho."""
    import math

    if abs(nu - 0.5) < 1e-9:
        return -math.log(rho)

    def P(z):
        if abs(nu - 1.5) < 1e-9:
            return 1.0 + z
        elif abs(nu - 2.5) < 1e-9:
            return 1.0 + z + (z**2)/3.0
        else:
            raise ValueError("nu must be 0.5, 1.5 or 2.5")

    def f(z):
        return P(z) * math.exp(-z) - rho

    z_lo, z_hi = 0.0, 50.0
    if f(z_lo) < 0:
        return z_lo

    while f(z_hi) > 0 and z_hi < 1e6:
        z_hi *= 2.0

    for _ in range(max_iter):
        z_mid = 0.5 * (z_lo + z_hi)
        if f(z_mid) > 0:
            z_lo = z_mid
        else:
            z_hi = z_mid
        if (z_hi - z_lo) < tol:
            break
    return 0.5 * (z_lo + z_hi)


def matern_c(nu: float) -> float:
    """Scaling constant for Matérn kernels: z = c * r / ell."""
    if abs(nu - 0.5) < 1e-9: return 1.0
    if abs(nu - 1.5) < 1e-9: return np.sqrt(3.0)
    if abs(nu - 2.5) < 1e-9: return np.sqrt(5.0)
    raise ValueError("nu must be 0.5, 1.5 or 2.5")


def estimate_ell_for_rhos(
    n=4000, rhos=(0.5, 0.8), nu=1.5, seed=42, max_pairs=200_000, noise_std=1.0
):
    """Estimate ell ranges by matching median distance to target correlations."""
    np.random.seed(seed)

    X_local, T_local, _ = generate_unified_data(n_samples=n, noise_std=noise_std)

    # Combine covariates X and treatment T for distance calculation
    Z = np.hstack([X_local, T_local.reshape(-1, 1)])

    r_med = pairwise_dist_median(Z, max_pairs=max_pairs, seed=seed)
    c_val = matern_c(nu)

    out = {"r_median": r_med, "nu": nu, "results": []}
    for rho in rhos:
        z_rho = solve_z_for_tau(rho, nu=nu)
        ell = (c_val * r_med) / z_rho
        out["results"].append({"rho": rho, "z_rho": z_rho, "ell_rho": ell})
    return out


In [7]:
# ============================================================
# Step 1: Estimate Median-based Length Parameters
# ============================================================
# This step helps determine the ell grid used in the next step.

heuristic_res = estimate_ell_for_rhos(
    n=500,
    rhos=(0.15, 0.5, 0.85),
    nu=1.5,
    seed=42,
)

print(f"r_median = {heuristic_res['r_median']:.6f}, nu = {heuristic_res['nu']}")
for r in heuristic_res["results"]:
    print(f"rho={r['rho']}: z_rho={r['z_rho']:.6f}, ell_rho={r['ell_rho']:.6f}")


r_median = 3.684261, nu = 1.5
rho=0.15: z_rho=3.372442, ell_rho=1.892198
rho=0.5: z_rho=1.678347, ell_rho=3.802150
rho=0.85: z_rho=0.683239, ell_rho=9.339821


In [8]:
# ============================================================
# Step 2: Generate Data and Run CV Grid Search
# ============================================================

# 1. Generate the dataset for the experiment
X, T, Y = generate_unified_data(
    n_samples=500,
    seed=42,
    noise_std=1,
)

print(f"Data Generated. Shapes -> X: {X.shape}, T: {T.shape}, Y: {Y.shape}")

# 2. Run LOOCV grid search
res = krr_length_selection_loocv_joint(
    Xs=X,
    Ts=T,                         # T is the treatment variable
    Ys=Y,
    nu_list=[1.5],               # Matérn smoothness
    ell_list=[2, 3, 4, 5, 6, 7, 8, 9],
    beta_bounds=(1e-4, 1e2),
    kernel_type="matern",
)

print("\nBest Global Parameter Set:")
print(res["best_global"])


Data Generated. Shapes -> X: (500, 10), T: (500,), Y: (500,)

Best Global Parameter Set:
{'kernel': 'matern', 'nu': 1.5, 'ell': 3, 'beta_star': 5.362741335545741, 'loocv_mse': 1.2982073751393195}


# T-to-Y Regression Length Parameter Selection

This section tunes a **T-only** KRR baseline using T as the sole input.


In [9]:
import numpy as np

def estimate_ell_for_rhos_T_only(
    n: int = 1000,
    rhos: tuple = (0.15, 0.85),
    nu: float = 1.5,
    seed: int = 42,
    max_pairs: int = 200_000,
    noise_std: float = 1.0,
):
    """Estimate ell values using only the treatment T (no covariates)."""
    np.random.seed(seed)

    _, T_local, _ = generate_unified_data(n_samples=n, noise_std=noise_std)

    Z = T_local.reshape(-1, 1)
    r_med = pairwise_dist_median(Z, max_pairs=max_pairs, seed=seed)
    c_val = matern_c(nu)

    out = {"r_median": float(r_med), "nu": float(nu), "results": []}
    for rho in rhos:
        z_rho = solve_z_for_tau(rho, nu=nu)
        ell = (c_val * r_med) / z_rho
        out["results"].append({"rho": float(rho), "z_rho": float(z_rho), "ell_rho": float(ell)})

    return out


# Example run
heur_T = estimate_ell_for_rhos_T_only(
    n=500,
    rhos=(0.15, 0.85),
    nu=1.5,
    seed=42,
    noise_std=1.0,
)

print(f"[T-only heuristic] r_median(T) = {heur_T['r_median']:.6f}, nu = {heur_T['nu']}")
for r in heur_T["results"]:
    print(f"rho={r['rho']}: z_rho={r['z_rho']:.6f}, ell_rho={r['ell_rho']:.6f}")

ell_low = [r["ell_rho"] for r in heur_T["results"] if abs(r["rho"] - 0.85) < 1e-12][0]
ell_high = [r["ell_rho"] for r in heur_T["results"] if abs(r["rho"] - 0.15) < 1e-12][0]
print(f"Suggested ell range (T-only, rho in [0.15, 0.85]): [{ell_low:.6f}, {ell_high:.6f}]")


[T-only heuristic] r_median(T) = 2.563962, nu = 1.5
rho=0.15: z_rho=3.372442, ell_rho=1.316824
rho=0.85: z_rho=0.683239, ell_rho=6.499797
Suggested ell range (T-only, rho in [0.15, 0.85]): [6.499797, 1.316824]


In [10]:
import numpy as np

# Generate a dataset for T-only tuning
X, T, Y = generate_unified_data(
    n_samples=500,
    seed=0,
    noise_std=1,
)

T = np.asarray(T).reshape(-1, 1)
Y = np.asarray(Y).reshape(-1)

print(f"Data generated. Shapes -> X: {np.asarray(X).shape}, T: {T.shape}, Y: {Y.shape}")


Data generated. Shapes -> X: (500, 10), T: (500, 1), Y: (500,)


In [11]:
import numpy as np
from typing import Dict, List, Tuple

def matern32_kernel_1d(T: np.ndarray, ell: float) -> np.ndarray:
    """Matérn(ν=1.5) kernel for 1D inputs."""
    T = np.asarray(T, dtype=float).reshape(-1, 1)
    r = np.abs(T - T.T)
    s = (np.sqrt(3.0) * r) / float(ell)
    return (1.0 + s) * np.exp(-s)


def loocv_mse_krr_from_kernel(K: np.ndarray, y: np.ndarray, beta: float) -> float:
    """Efficient LOOCV MSE for KRR with a fixed kernel matrix K."""
    y = np.asarray(y, dtype=float).reshape(-1)
    n = K.shape[0]
    M = K + float(beta) * np.eye(n, dtype=float)

    L = np.linalg.cholesky(M)

    tmp = np.linalg.solve(L, y)
    alpha = np.linalg.solve(L.T, tmp)

    V = np.linalg.solve(L, np.eye(n, dtype=float))
    diagC = np.sum(V * V, axis=0)

    residual = alpha / diagC
    return float(np.mean(residual ** 2))


def make_beta_grid(beta_bounds: Tuple[float, float], num_beta: int) -> np.ndarray:
    """Create a log-spaced beta grid within [beta_min, beta_max]."""
    beta_min, beta_max = beta_bounds
    beta_min = float(beta_min)
    beta_max = float(beta_max)
    if beta_min <= 0 or beta_max <= 0 or beta_min >= beta_max:
        raise ValueError("beta_bounds must satisfy 0 < beta_min < beta_max.")
    return np.logspace(np.log10(beta_min), np.log10(beta_max), num=num_beta)


def select_ell_by_loocv_with_inner_beta_search(
    T: np.ndarray,
    Y: np.ndarray,
    ell_list: List[float],
    beta_bounds: Tuple[float, float] = (1e-4, 1e2),
    num_beta: int = 30,
    verbose: bool = True,
) -> Dict:
    """Outer loop over ell; inner loop over beta (LOOCV MSE)."""
    T = np.asarray(T, dtype=float).reshape(-1, 1)
    Y = np.asarray(Y, dtype=float).reshape(-1)

    beta_grid = make_beta_grid(beta_bounds, num_beta=num_beta)

    per_ell_results = []
    best_global = {"ell": None, "beta": None, "loocv_mse": np.inf}

    for ell in ell_list:
        K = matern32_kernel_1d(T, ell=float(ell))

        best_beta_for_ell = None
        best_mse_for_ell = np.inf

        for beta in beta_grid:
            mse = loocv_mse_krr_from_kernel(K, Y, beta=float(beta))
            if mse < best_mse_for_ell:
                best_mse_for_ell = mse
                best_beta_for_ell = float(beta)

        per_ell_results.append(
            {"ell": float(ell), "best_beta": float(best_beta_for_ell), "best_loocv_mse": float(best_mse_for_ell)}
        )

        if verbose:
            print(f"ell={float(ell):>8.4f} | best_beta={best_beta_for_ell:.4e} | best_LOOCV_MSE={best_mse_for_ell:.6e}")

        if best_mse_for_ell < best_global["loocv_mse"]:
            best_global = {"ell": float(ell), "beta": float(best_beta_for_ell), "loocv_mse": float(best_mse_for_ell)}

    return {
        "best_global": best_global,
        "per_ell_results": per_ell_results,
        "beta_bounds": tuple(map(float, beta_bounds)),
        "num_beta": int(num_beta),
        "ell_list": [float(e) for e in ell_list],
    }


# Example run: initial ell grid
ell_grid = [1, 2, 3, 4, 5, 6, 7]

res_T_only = select_ell_by_loocv_with_inner_beta_search(
    T=T,
    Y=Y,
    ell_list=ell_grid,
    beta_bounds=(1e-4, 1e2),
    num_beta=30,
    verbose=True,
)

print("\nBest (T-only) global parameters:")
print(res_T_only["best_global"])


ell=  1.0000 | best_beta=2.2122e+00 | best_LOOCV_MSE=9.963566e-01
ell=  2.0000 | best_beta=5.2983e-01 | best_LOOCV_MSE=9.961879e-01
ell=  3.0000 | best_beta=1.2690e-01 | best_LOOCV_MSE=9.963176e-01
ell=  4.0000 | best_beta=7.8805e-02 | best_LOOCV_MSE=9.964268e-01
ell=  5.0000 | best_beta=3.0392e-02 | best_LOOCV_MSE=9.964353e-01
ell=  6.0000 | best_beta=1.8874e-02 | best_LOOCV_MSE=9.964684e-01
ell=  7.0000 | best_beta=1.1721e-02 | best_LOOCV_MSE=9.964966e-01

Best (T-only) global parameters:
{'ell': 2.0, 'beta': 0.5298316906283708, 'loocv_mse': 0.9961879016900578}
